In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Compute the 2D Discrete Fourier Transform (2D DFT) of a complex-valued signal stored on the GPU.
  Given a 2D complex input signal of shape <code>M &times; N</code>, compute its 2D DFT spectrum
  using the row-column decomposition: apply a 1D DFT along each row, then a 1D DFT along each
  column of the result. All values are 32-bit floating point.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>spectrum</code></li>
  <li>
    The input and output are stored as 1D arrays of interleaved real and imaginary parts in
    row-major order: element <code>x[m, n]</code> has its real part at index
    <code>2*(m*N + n)</code> and imaginary part at index <code>2*(m*N + n) + 1</code>
  </li>
</ul>

<h2>Example</h2>
<p>
Input: <code>M</code> = 2, <code>N</code> = 2<br>
Signal $x[m, n]$ (real part):
$$
\begin{bmatrix}
1.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
Signal $x[m, n]$ (imaginary part):
$$
\begin{bmatrix}
0.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
Output:<br>
Spectrum $X[k, l]$ (real part):
$$
\begin{bmatrix}
1.0 & 1.0 \\
1.0 & 1.0
\end{bmatrix}
$$
Spectrum $X[k, l]$ (imaginary part):
$$
\begin{bmatrix}
0.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 4096</li>
  <li>Signal values are 32-bit floating point (real and imaginary parts)</li>
  <li>Performance is measured with <code>M</code> = 2,048, <code>N</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// signal, spectrum are device pointers
extern "C" void solve(const float* signal, float* spectrum, int M, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# signal, spectrum are tensors on the GPU
@cute.jit
def solve(signal: cute.Tensor, spectrum: cute.Tensor, M: cute.Int32, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# signal is a tensor on GPU
@jax.jit
def solve(signal: jax.Array, M: int, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# signal, spectrum are device pointers
@export
def solve(
    signal: UnsafePointer[Float32, MutExternalOrigin],
    spectrum: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# signal, spectrum are tensors on the GPU
def solve(signal: torch.Tensor, spectrum: torch.Tensor, M: int, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# signal, spectrum are tensors on the GPU
def solve(signal: torch.Tensor, spectrum: torch.Tensor, M: int, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="2D FFT",
            atol=1e-02,
            rtol=1e-02,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(self, signal: torch.Tensor, spectrum: torch.Tensor, M: int, N: int):
        assert signal.shape == (M * N * 2,)
        assert spectrum.shape == (M * N * 2,)
        assert signal.dtype == torch.float32
        assert spectrum.dtype == torch.float32
        assert signal.device == spectrum.device

        sig_ri = signal.view(M, N, 2)
        sig_c = torch.complex(sig_ri[..., 0].contiguous(), sig_ri[..., 1].contiguous())
        spec_c = torch.fft.fft2(sig_c)
        spec_ri = torch.stack((spec_c.real, spec_c.imag), dim=-1).contiguous()
        spectrum.copy_(spec_ri.view(-1))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "signal": (ctypes.POINTER(ctypes.c_float), "in"),
            "spectrum": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M, N = 2, 2
        signal = torch.tensor([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], device="cuda", dtype=dtype)
        spectrum = torch.empty(M * N * 2, device="cuda", dtype=dtype)
        return {"signal": signal, "spectrum": spectrum, "M": M, "N": N}

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        cases = []

        def make_case(M, N, low=-1.0, high=1.0):
            signal = torch.empty(M * N * 2, device="cuda", dtype=dtype).uniform_(low, high)
            spectrum = torch.empty(M * N * 2, device="cuda", dtype=dtype)
            return {"signal": signal, "spectrum": spectrum, "M": M, "N": N}

        def make_zero_case(M, N):
            signal = torch.zeros(M * N * 2, device="cuda", dtype=dtype)
            spectrum = torch.empty(M * N * 2, device="cuda", dtype=dtype)
            return {"signal": signal, "spectrum": spectrum, "M": M, "N": N}

        def make_impulse_case(M, N):
            signal = torch.zeros(M * N * 2, device="cuda", dtype=dtype)
            signal[0] = 1.0
            spectrum = torch.empty(M * N * 2, device="cuda", dtype=dtype)
            return {"signal": signal, "spectrum": spectrum, "M": M, "N": N}

        # Edge cases: small sizes
        cases.append(make_impulse_case(1, 1))
        cases.append(make_zero_case(2, 2))
        cases.append(make_case(1, 4))

        # Power-of-2 sizes
        cases.append(make_case(16, 16))
        cases.append(make_case(32, 64))

        # Non-power-of-2 sizes
        cases.append(make_case(3, 5))
        cases.append(make_case(30, 30))

        # Mixed positive/negative values
        cases.append(make_case(100, 200, low=-5.0, high=5.0))

        # Realistic sizes
        cases.append(make_case(256, 256))
        cases.append(make_case(512, 512))

        return cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M, N = 2048, 2048
        signal = torch.empty(M * N * 2, device="cuda", dtype=dtype).normal_(0.0, 1.0)
        spectrum = torch.empty(M * N * 2, device="cuda", dtype=dtype)
        return {"signal": signal, "spectrum": spectrum, "M": M, "N": N}


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
